# 長江杯 2026 — A100 Colab 自動跑批與自癒斷點續跑 Worker（Ultra 5x 1,000 CU 省額度版）

**核心防中斷與自癒特性**：
- 🔍 **第 1 格自動診斷 (`--diagnose`)**：一鍵檢查 Google Drive 上每個階段（`s1_mono`, `s23`, `s1_trimer`, `s23_var`）已完成的 PDB 數量與 `worker.log` 最後 12 行記錄。
- 🔄 **第 4 格自動找回未完任務 (`[AUTO-RECOVER]`)**：即使上次異常斷線把任務單移進了 `jobs/completed/` 或 `jobs/failed/`，只要檢測到有任一階段未滿 100%，第 4 格會**自動把任務單拉回 `jobs/running/` 並從斷點繼續跑**（已算完的序列 0 秒跳過）！
- 🛡️ **智慧釋放 GPU（15 秒倒數）**：僅當所有階段 100% 驗證完成後才倒數 15 秒執行 `runtime.unassign()`。

In [1]:
#@title 1. 檢查 GPU 架構、掛載共享 Google Drive (1NPtj9CpPoq1JJX42EZbdp7wfdQ8THsQg)、同步 GitHub 並印出診斷報告
import os, sys, subprocess, pathlib, importlib, torch
from google.colab import drive

SHARED_DRIVE_URL = "https://drive.google.com/drive/folders/1NPtj9CpPoq1JJX42EZbdp7wfdQ8THsQg?usp=drive_link" #@param {type:"string"}
GITHUB_REPO_URL = "https://github.com/David-Liv/changjiang-protein-2026.git" #@param {type:"string"}
REPO_DIR = "/content/changjiang-protein-2026"

print("=== [1/4] GPU 架構與 1,000 CU 額度自檢 ===")
!nvidia-smi
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    gb = p.total_memory / 1e9
    sm = f"sm_{p.major}{p.minor}"
    print(f"檢測到 GPU: {p.name} | 顯存: {gb:.1f} GB | 架構: {sm}")
    if p.major >= 12 or "BLACKWELL" in p.name.upper() or "RTX PRO 6000" in p.name.upper():
        print("❌ [警告] 當前為 G4 (Blackwell sm_120)！BindCraft (CUDA 12.4/JAX) 不支持此架構，請切換為 A100！")
    elif "H100" in p.name.upper() or p.major == 9:
        print("⚠️ [提示] 當前為 H100 (~18 CU/h，1,000 CU 約可跑 55 小時)。日常跑批建議改選 A100 以節省額度。")
    else:
        print("✅ [最佳配置] A100 架構完全兼容 ESMFold 與 BindCraft（~5.4–7.5 CU/h，1,000 CU 可跑 135–185 小時）✓")
else:
    print("❌ 未檢測到 GPU！請在『執行階段 -> 變更執行階段類型』選擇 A100 GPU。")

print("\n=== [2/4] 檢查 ESMFold / ipSAE 評估依賴（不升級 numpy，防止內核崩潰）===")
!pip install -q transformers accelerate

print("\n=== [3/4] 掛載 Google Drive（同一執行階段只驗證 1 次，永不重複卸載）===")
if not os.path.ismount('/content/drive') and not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')
else:
    print("✅ Google Drive 已處於掛載狀態，免重複驗證！")

print("\n=== [4/4] 同步 GitHub 最新代碼並直連團隊共享資料夾 (ID: 1NPtj9CpPoq1JJX42EZbdp7wfdQ8THsQg) ===")
if not os.path.exists(REPO_DIR):
    !git clone {GITHUB_REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull --ff-only

%cd {REPO_DIR}
for p in [REPO_DIR, f"{REPO_DIR}/pipeline", f"{REPO_DIR}/scripts"]:
    if p not in sys.path:
        sys.path.insert(0, p)

import worker
importlib.reload(worker)
# 嚴格鎖定共享資料夾 ID，絕不在私人 MyDrive 自動 mkdir 新資料夾！
DRIVE_WORKSPACE = str(worker.resolve_shared_drive_workspace(SHARED_DRIVE_URL))
print(f"🔗 已鎖定團隊共享網盤實際路徑：{DRIVE_WORKSPACE}")

!python3 scripts/tnf_pipeline.py selftest
worker.main(["--drive-root", DRIVE_WORKSPACE, "--diagnose"])

=== [1/4] GPU 架構與 1,000 CU 額度自檢 ===
Fri Nov  6 08:02:03 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          Off |   00000000:00:05.0 Off |                    0 |
| N/A   33C    P0             55W /  400W |       0MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------

2

In [1]:
#@title 2. 安裝 BindCraft + PyRosetta + SolubleMPNN 環境（僅跑靶點 1 / 5 生成任務時需要；若只跑 TNF-α 校準請跳過此格！）
import os, torch
if torch.cuda.is_available() and torch.cuda.get_device_properties(0).major >= 12:
    raise RuntimeError("當前 GPU 為 G4 (Blackwell sm_120)，與 BindCraft CUDA 12.4 不兼容！請先切換執行階段為 A100 再安裝。")

if not os.path.exists("/content/miniforge3/bin/conda"):
    print("正在安裝 Miniforge3（不重啟內核）...")
    !wget -q -O /content/Miniforge3.sh https://github.com/conda-forge/miniforge/releases/latest/download/Miniforge3-Linux-x86_64.sh
    !bash /content/Miniforge3.sh -b -p /content/miniforge3
os.environ["PATH"] = "/content/miniforge3/bin:" + os.environ.get("PATH", "")

if not os.path.exists("/content/BindCraft"):
    !git clone https://github.com/martinpacesa/BindCraft /content/BindCraft
    !bash /content/BindCraft/install_bindcraft.sh --cuda '12.4' --pkg_manager 'conda'
else:
    print("BindCraft 已安裝 ✓")

正在安裝 Miniforge3（不重啟內核）...
PREFIX=/content/miniforge3
Unpacking bootstrapper...
Unpacking payload...
Extracting ca-certificates-2026.10.23-hbd8a1cb_0.conda
Extracting libgomp-16.2.0-he0feb66_7.conda
Extracting libzlib-1.3.2-h25fd6f3_3.conda
Extracting nlohmann_json-abi-3.12.0-h0f90c79_2.conda
Extracting pybind11-abi-11-hc364b38_1.conda
Extracting python_abi-3.14-9_cp314.conda
Extracting tzdata-2026e-h5d301ff_0.conda
Extracting _openmp_mutex-4.5-20_gnu.conda
Extracting zstd-1.5.7-hb78ec9c_7.conda
Extracting ld_impl_linux-64-2.46.1-default_hbd61a6d_103.conda
Extracting libgcc-16.2.0-ha9f2e26_7.conda
Extracting bzip2-1.0.8-hda65f42_10.conda
Extracting c-ares-1.34.9-hebe6cf0_0.conda
Extracting keyutils-1.6.3-h7cc23a3_1.conda
Extracting libev-4.33-h280c20c_3.conda
Extracting libexpat-2.9.0-hd2095e1_0.conda
Extracting libffi-3.7.0-h81df57d_1.conda
Extracting libiconv-1.18-h0cb94f2_3.conda
Extracting liblzma-5.8.3-hb03c661_1.conda
Extracting libmpdec-4.0.0-hb03c661_2.conda
Extracting libpython

In [3]:
#@title 3. 將預設任務模板投遞到共享網盤 `jobs/pending/`（預設已切為 Target 5 TL1A / Target 1 EGFR 生成任務！）
import shutil, pathlib, worker

TEMPLATE_CHOICE = "target5_TL1A_minibinder" #@param ["target5_TL1A_minibinder", "target1_EGFR_vhh", "target2_TNFa_calibration", "none"]
shared_root = worker.resolve_shared_drive_workspace(SHARED_DRIVE_URL)
pending_dir = shared_root / "jobs" / "pending"
running_dir = shared_root / "jobs" / "running"
pending_dir.mkdir(parents=True, exist_ok=True)
running_dir.mkdir(parents=True, exist_ok=True)

mapping = {
    "target5_TL1A_minibinder": "/content/changjiang-protein-2026/jobs/templates/job_target5_TL1A_minibinder.json",
    "target1_EGFR_vhh": "/content/changjiang-protein-2026/jobs/templates/job_target1_EGFR_vhh.json",
    "target2_TNFa_calibration": "/content/changjiang-protein-2026/jobs/templates/job_target2_TNFa_calibration.json",
}
if TEMPLATE_CHOICE in mapping:
    src = pathlib.Path(mapping[TEMPLATE_CHOICE])
    if (running_dir / src.name).exists():
        print(f"🔄 任務 {src.name} 已在共享網盤 jobs/running/ 中，直接運行『第 4 格』即可從斷點繼續！")
    else:
        dst = pending_dir / src.name
        shutil.copy2(src, dst)
        print(f"✅ 已投遞任務單至團隊共享網盤隊列：{dst}")

✅ 已投遞任務單至團隊共享網盤隊列：/content/drive/.shortcut-targets-by-id/1NPtj9CpPoq1JJX42EZbdp7wfdQ8THsQg/Changjiang_Cup_2026/jobs/pending/job_target5_TL1A_minibinder.json


In [4]:
!pip install matplotlib-venn

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.0/11.0 MB 114.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 160.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 73.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 178.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.1/7.1 MB 188.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.3/35.3 MB 198.5 MB/s  0:00:00
  Created wheel for matplotlib-venn: filename=matplotlib_venn-1.1.2-py3-none-any.whl size=45388 sha256=7bd75d5f8f641dbf31a829bc2a80f71eb9ce7facb1cb0f58f0c7e715a8c6a737
  Stored in directory: /root/.cache/pip/wheels/0b/00/0b/21afd129ca9ca45bd4b9657891344063af59e81ec2640cf4ef
Successfully built matplotlib-venn
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12/12 [matplotlib-venn]


In [25]:
# https://pypi.python.org/pypi/libarchive-c
!apt-get -qq install -y libarchive-dev
!/usr/bin/python3 -m pip install -U libarchive-c
import libarchive

  Using cached libarchive_c-5.3-py3-none-any.whl.metadata (5.8 kB)
Using cached libarchive_c-5.3-py3-none-any.whl (17 kB)


In [28]:
#@title 4. 啟動隊列執行器（顯存常駐 + 同長度 Batch 並行加速 + 永不卸載網盤免重複驗證）
import os, sys, time, importlib
from google.colab import runtime

# 提示：若你要連續跑多個任務，請取消勾選 AUTO_UNASSIGN_GPU，這樣不會斷開 A100 也不需重新驗證網盤！
# 若你要掛機睡覺/離開，請勾選 AUTO_UNASSIGN_GPU，跑完 100% 後會自動釋放 A100 省額度。
AUTO_UNASSIGN_GPU = True #@param {type:"boolean"}
os.environ["PATH"] = "/content/miniforge3/bin:" + os.environ.get("PATH", "")

if not os.path.exists("/content/changjiang-protein-2026"):
    !git clone https://github.com/David-Liv/changjiang-protein-2026.git /content/changjiang-protein-2026
else:
    !git -C /content/changjiang-protein-2026 pull --ff-only

for p in ["/content/changjiang-protein-2026", "/content/changjiang-protein-2026/pipeline", "/content/changjiang-protein-2026/scripts"]:
    if p not in sys.path:
        sys.path.insert(0, p)

import tnf_pipeline as tp
import worker
importlib.reload(tp)
importlib.reload(worker)

DRIVE_WORKSPACE = str(worker.resolve_shared_drive_workspace(SHARED_DRIVE_URL))
argv = ["--drive-root", DRIVE_WORKSPACE, "--once"]
if AUTO_UNASSIGN_GPU:
    argv.append("--unassign-on-exit")

# 在 Colab 主 IPython Kernel 內直接執行（不呼叫 drive.flush_and_unmount()，避免下次要重新驗證 Google Drive！）
exit_code = worker.main(argv)

if AUTO_UNASSIGN_GPU and exit_code == 0:
    if hasattr(os, "sync"):
        os.sync()
    print("🔌 正在從 Colab 主內核強制釋放 A100 GPU (runtime.unassign)...", flush=True)
    time.sleep(2)
    runtime.unassign()

Already up to date.


ModuleNotFoundError: No module named 'tnf_pipeline'